# Imports

In [9]:
import random
from typing import Optional
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Agent

In [48]:
class RL_Bandit_Agent:
    def __init__(self):
        self.alpha = 0.1
        self.B = 1.0
        self.gamma = 0.95
        self.Q_sa = np.zeros((8, 8, 2, 2, 2, 2, 4))
        self.action = random.choice([0, 1, 2, 3])

    def softmax(self, Q_sa):
        Q_sa = Q_sa - np.max(Q_sa)
        exp_vals = np.exp(self.B * Q_sa)
        return exp_vals / np.sum(exp_vals)

    def take_action(self, x, y, hole_features):
        hr, hu, hl, hd = hole_features
        probs = self.softmax(self.Q_sa[x, y, hr, hu, hl, hd])
        self.action = np.random.choice(len(probs), p=probs)
        return self.action

    def update_Q(self, reward, x, y, hole_features, nx, ny, new_hole_features):
        hr, hu, hl, hd = hole_features
        nhr, nhu, nhl, nhd = new_hole_features
        future_value = np.max(self.Q_sa[nx, ny, nhr, nhu, nhl, nhd])
        target = reward + self.gamma * future_value
        self.Q_sa[x, y, hr, hu, hl, hd, self.action] += self.alpha * (
            target - self.Q_sa[x, y, hr, hu, hl, hd, self.action]
        )

# Env

In [49]:
class GridWorldEnv(gym.Env):
    def __init__(self, size: int = 8, hole_coords=None):
        self.size = size
        self._agent_location = np.array([-1, -1], dtype=np.int32)
        self._target_location = np.array([-1, -1], dtype=np.int32)
        self.prev_location = None

        self.observation_space = gym.spaces.Dict({
            "agent": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),
            "target": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),
            "hole_features": gym.spaces.Box(0, 1, shape=(4,), dtype=int),
        })
        self.action_space = gym.spaces.Discrete(4)

        self._action_to_direction = {
            0: np.array([0, 1]),   # right
            1: np.array([-1, 0]),  # up
            2: np.array([0, -1]),  # left
            3: np.array([1, 0]),   # down
        }

        self.hole_coords = hole_coords if hole_coords is not None else {(1, 1), (3, 1), (3, 2)}
        self._build_hole_map()

    def _build_hole_map(self):
        self.hole_map = np.zeros((self.size, self.size), dtype=int)
        for (r, c) in self.hole_coords:
            self.hole_map[r, c] = 1

    def _get_local_hole_features(self):
        x, y = self._agent_location
        features = []
        for action_idx in range(4):
            d = self._action_to_direction[action_idx]
            nx, ny = x + d[0], y + d[1]
            if 0 <= nx < self.size and 0 <= ny < self.size:
                features.append(self.hole_map[nx, ny])
            else:
                features.append(0)
        return tuple(features)

    def _get_obs(self):
        return {
            "agent": self._agent_location,
            "target": self._target_location,
            "hole_features": self._get_local_hole_features(),
        }

    def _get_info(self):
        return {"distance": np.linalg.norm(self._agent_location - self._target_location, ord=1)}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._agent_location = np.array([0, 0], dtype=np.int32)
        self._target_location = np.array([7, 7], dtype=np.int32)

        if options and "hole_coords" in options:
            self.hole_coords = options["hole_coords"]
            self._build_hole_map()

        self.prev_location = self._agent_location.copy()
        return self._get_obs(), self._get_info()

    def step(self, action):
        direction = self._action_to_direction[action]
        self.prev_location = self._agent_location.copy()
        self._agent_location = np.clip(self._agent_location + direction, 0, self.size - 1)

        terminated = np.array_equal(self._agent_location, self._target_location)
        truncated = False

        location_as_tuple = tuple(self._agent_location)
        if terminated:
            reward = 10
        elif location_as_tuple in self.hole_coords:
            reward = -10
        else:
            reward = -0.1  # Small step cost to encourage short paths

        return self._get_obs(), reward, terminated, truncated, self._get_info()



# Training Loop

In [50]:
env = GridWorldEnv()
agent = RL_Bandit_Agent()

for episode in range(5000):
    num_holes = random.randint(1, 6)
    holes = set()
    while len(holes) < num_holes:
        coord = (random.randint(0, 7), random.randint(0, 7))
        if coord != (0, 0) and coord != (7, 7):
            holes.add(coord)

    env.reset(options={"hole_coords": holes})
    terminated = False

    for step in range(300):
        if not terminated:
            hole_features = env._get_local_hole_features()
            loc = env._agent_location.copy()
            action = agent.take_action(loc[0], loc[1], hole_features)
            obs, reward, terminated, truncated, info = env.step(action)
            new_hole_features = env._get_local_hole_features()
            new_loc = env._agent_location
            agent.update_Q(reward, loc[0], loc[1], hole_features,
                          new_loc[0], new_loc[1], new_hole_features)

    if (episode + 1) % 1000 == 0:
        print(f"Episode {episode + 1}/5000 done")

print("Training complete")

Episode 1000/5000 done
Episode 2000/5000 done
Episode 3000/5000 done
Episode 4000/5000 done
Episode 5000/5000 done
Training complete


# Animation

In [38]:
fig, ax = plt.subplots(figsize=(6, 6))
target = tuple(env._target_location)
holes = env.hole_coords

def animate(i):
    ax.clear()
    ax.set_xlim(-0.5, env.size - 0.5)
    ax.set_ylim(-0.5, env.size - 0.5)
    ax.set_xticks(range(env.size))
    ax.set_yticks(range(env.size))
    ax.grid(True)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_title(f'Step {i} / {len(path)-1}')

    # Holes (red)
    for (r, c) in holes:
        ax.plot(c, r, 's', color='red', markersize=18)

    # Target (green star)
    ax.plot(target[1], target[0], '*', color='green', markersize=22)

    # Agent trail (light blue dots)
    for j in range(max(0, i - 10), i):
        alpha = 0.15 + 0.05 * (j - max(0, i - 10))
        ax.plot(path[j][1], path[j][0], 'o', color='cornflowerblue', markersize=8, alpha=alpha)

    # Agent (blue square)
    ax.plot(path[i][1], path[i][0], 's', color='blue', markersize=18)

anim = FuncAnimation(fig, animate, frames=len(path), interval=150)
plt.close()
HTML(anim.to_jshtml())


# Testing

In [61]:
agent.B = 5.0  # High confidence at test time
new_holes = {(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)}
env.reset(options={"hole_coords": new_holes})
terminated = False
test_path = [tuple(env._agent_location)]

for step in range(500):
    if not terminated:
        hole_features = env._get_local_hole_features()
        loc = env._agent_location.copy()
        action = agent.take_action(loc[0], loc[1], hole_features)
        obs, reward, terminated, truncated, info = env.step(action)
        test_path.append(tuple(env._agent_location))

print(f"Finished in {len(test_path)-1} steps, reached target: {terminated}")


Finished in 28 steps, reached target: True


In [62]:
fig, ax = plt.subplots(figsize=(6, 6))
target = tuple(env._target_location)
holes = env.hole_coords

def animate(i):
    ax.clear()
    ax.set_xlim(-0.5, env.size - 0.5)
    ax.set_ylim(-0.5, env.size - 0.5)
    ax.set_xticks(range(env.size))
    ax.set_yticks(range(env.size))
    ax.grid(True)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_title(f'Test - Step {i} / {len(test_path)-1}')

    for (r, c) in holes:
        ax.plot(c, r, 's', color='red', markersize=18)
    ax.plot(target[1], target[0], '*', color='green', markersize=22)

    for j in range(max(0, i - 10), i):
        alpha = 0.15 + 0.05 * (j - max(0, i - 10))
        ax.plot(test_path[j][1], test_path[j][0], 'o', color='cornflowerblue', markersize=8, alpha=alpha)

    ax.plot(test_path[i][1], test_path[i][0], 's', color='blue', markersize=18)

anim = FuncAnimation(fig, animate, frames=len(test_path), interval=150)
plt.close()
HTML(anim.to_jshtml())
